# Document ingestion and chunking

When working with raw text data, we need to do several tasks before storing the text data for retrieval:
* import the text from a file (PDF, TXT, Markdown, ...)
* chunk the text
* generate embedding vector for each chunk
* potentially combine the text and embedding with metadata into one chunk object

# Embedding

We start with embedding here because we have seen this before, both with an API and with a local model.

In [ ]:
from openai import OpenAI

In [ ]:
import os
NRP_TOK = os.environ.get('NRP_TOK')

In [ ]:
client = OpenAI(api_key = NRP_TOK,
                base_url = "https://ellm.nrp-nautilus.io/v1")

Here we set the model, but note that this is an **embedding** model rather than a chat model.

In [ ]:
nrp_hosted_model = 'embed-mistral'

To generate the embeddings, we use `client.embeddings.create` rather than the previous client endpoint we've used before (`client.chat.completions.create`)

In [ ]:
response = client.embeddings.create(model=nrp_hosted_model,
                                    input="What's our refund policy for annual subscriptions?",
                                    )

In [ ]:
len(response.data[0].embedding)

In [ ]:
def embed(text):
    response = client.embeddings.create(
        model=nrp_hosted_model,
        input=text,
    )
    return response.data[0].embedding

query = "What's our refund policy for annual subscriptions?"
query_embedding = embed(query)
print(len(query_embedding))  # e.g., 4096

In [ ]:
# query_embedding

This sends your text off to the API for embedding.  If we really want to be self-contained (i.e. have sensitive or private data), then we can alternatively use a local model to do this:

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

Load a local embedding model (downloaded once, then used locally)
* You can swap this for any other SentenceTransformer-compatible model.

In [ ]:
LOCAL_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(LOCAL_MODEL_NAME)

In [ ]:
query = "What's our refund policy for annual subscriptions?"

In [ ]:
vec = embedding_model.encode([query], convert_to_numpy=True)

In [ ]:
vec.shape

In [ ]:
def embed(text):

    # sentence-transformers can take a list; we wrap single text for simplicity
    vec = embedding_model.encode([text], convert_to_numpy=True)[0]
    
    # Convert numpy array to list to mirror the API-style output
    return vec.tolist()

In [ ]:
query = "What's our refund policy for annual subscriptions?"
query_embedding = embed(query)
print(len(query_embedding))  # e.g., 4096... or 384

In [ ]:
# query_embedding

Note the following:
* The LLM model is a specialized embedding model, not the chat model.
* The output is a list of floats.
* You usually store this in a vector DB along with metadata.

In [ ]:
texts = [
    "I love machine learning.",
    "Neural networks are fascinating.",
    "The cat sat on the mat.",
]

embeddings = [embed(t) for t in texts]
print(len(embeddings[0]))  # embedding dimension

In [ ]:
len(embeddings)

# Semantic similarity

Though assessing similarity is not one of our steps for ingestion and chunking, here we can look at how word embedding vectors might be used to assess text similarity

In [ ]:
from sentence_transformers import util

In [ ]:
docs = [
    "Customers can request a money-back guarantee within 30 days of purchase.",
    "Our support team is available 24/7 via chat and email.",
    "The annual subscription grants access to all premium features."
]

query = "What is the refund policy?"

# Compute embeddings
doc_embs = embedding_model.encode(docs, convert_to_tensor=True)
query_emb = embedding_model.encode(query, convert_to_tensor=True)

# Compute cosine similarities
# cosine_scores = util.cos_sim(query_emb, doc_embs)[0]
# or:
cosine_scores = query_emb @ doc_embs.T

for doc, score in zip(docs, cosine_scores):
    print(f"{score:.3f} :: {doc}")

Even though the query doesn’t contain the word "money-back" or "guarantee", that doc will likely have the highest score, because "refund policy" and "money‑back guarantee" are semantically similar.

In [ ]:
doc_embs.shape

In [ ]:
query_emb.shape

# Expanding to ingestion and chunking of documents

We're aiming to get something like the following representation of (chunks of) our documents:

```python
{
    "text": "full raw text here...",
    "metadata": {
        "source": "path/or/url",
        "type": "pdf/markdown/html",
        ...
    }
}


As examples, we'll use the following docs from our course materials:
* `LLM_Syllabus.pdf`: course syllabus as PDF
* `LLM_CoursePage.html`: UCLA Extension web page description for our course, taken from:
  * https://www.uclaextension.edu/computer-science/machine-learning-ai/course/large-language-models-com-sci-x-45046

In [ ]:
# To read in PDFs, with one "doc" per page

from pathlib import Path
from pypdf import PdfReader

def load_pdf(path):
    reader = PdfReader(path)
    docs = []
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = text.strip()
        if not text:
            continue
        docs.append({
            "text": text,
            "metadata": {
                "source": str(Path(path).name),
                "type": "pdf",
                "page": page_num,
            }
        })
    return docs

Each page becomes one "raw document" with page metadata.

In [ ]:
syllabus_docs = load_pdf('./LLM_Syllabus.pdf')
syllabus_docs[2]

In [ ]:
# To read in HTML

from pathlib import Path
from bs4 import BeautifulSoup

def load_html(path):
    html = Path(path).read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")

    main_text = soup.get_text("\n", strip=True)
    title = soup.title.string.strip() if soup.title and soup.title.string else None

    return [{
        "text": main_text,
        "metadata": {
            "source": str(Path(path).name),
            "type": "html",
            "title": title,
        }
    }]

In [ ]:
load_html('./LLM_CoursePage.html')

If we have functions like the above to handle different file types, we might combine them with the following that will read text from a variety of different documents, including TXT files as the default type.

In [ ]:
def load_documents(paths):
    all_docs = []
    for path in paths:
        ext = Path(path).suffix.lower()
        if ext == ".pdf":
            all_docs.extend(load_pdf(path))
        elif ext in {".md", ".markdown"}:
            all_docs.extend(load_markdown(path))
        elif ext in {".html", ".htm"}:
            all_docs.extend(load_html(path))
        else:
            # plain text fallback
            text = Path(path).read_text(encoding="utf-8")
            all_docs.append({
                "text": text,
                "metadata": {
                    "source": str(Path(path).name),
                    "type": "text",
                }
            })
    return all_docs

In [ ]:
raw_docs = load_documents(["LLM_Syllabus.pdf",
                           "LLM_CoursePage.html"])

In [ ]:
raw_docs[0]

# Chunking

Now that we have objects for text imported from our documents, we can break that up into pieces (chunks).

The following chunking code gets chunks for a single text.  Note also that it counts by characters, rather than tokens.

In [ ]:
def chunk_text(
    text,
    chunk_size = 800,      # characters (roughly ~200 tokens)
    chunk_overlap = 200    # characters (~50 tokens)
):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        # move start forward, but with overlap
        start += chunk_size - chunk_overlap

    return chunks

Different texts can be of different sizes and have different numbers of chunks.  This function will take a collection of documents and return a collective list of chunks for all of them, with metadata included for the unique `id` and `chunk_index` of each.

In [ ]:
def chunk_documents(
    docs,
    chunk_size = 800,
    chunk_overlap = 200
):
    chunked_docs = []
    count = 0
    
    for doc in docs:
        base_text = doc["text"]
        base_meta = doc["metadata"]

        for i, chunk in enumerate(chunk_text(base_text, chunk_size, chunk_overlap)):
            if not chunk:
                continue
            chunked_docs.append({
                "id": count,
                "text": chunk,
                "metadata": {
                    **base_meta,
                    "chunk_index": i,
                }
            })
            count += 1

    return chunked_docs

In [ ]:
raw_docs = load_documents(["LLM_Syllabus.pdf",
                           "LLM_CoursePage.html"])

chunks = chunk_documents(raw_docs, chunk_size=1000, chunk_overlap=200)

print(len(chunks), "chunks")
print(chunks[0]["metadata"], chunks[0]["text"][:200], "...")


In [ ]:
print(chunks[34]["metadata"], chunks[34]["text"])

In [ ]:
print(chunks[35]["metadata"], chunks[35]["text"])

# Combining

We have the functionality now to ingest documents, separate the text into chunks, generate embeddings, and tie together the text + embedding + metadata.

In [ ]:
raw_docs = load_documents(["LLM_Syllabus.pdf",
                           "LLM_CoursePage.html"])

chunks = chunk_documents(raw_docs, chunk_size=1000, chunk_overlap=200)

for i,chunk in enumerate(chunks):
    embedding = embed(chunk['text'])
    chunks[i]['embedding'] = embedding

print(len(chunks), "chunks")
print(chunks[0]["metadata"])
print(chunks[0]["text"][:200], "...")
print(chunks[0]["embedding"][:20], "...")

In [ ]:
chunks[35]